<a href="https://colab.research.google.com/github/geethika005/construction-risk/blob/main/paper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
df = pd.read_csv(url)

# Standard ProPublica cleaning filters
df = df[(df.days_b_screening_arrest <= 30) &
        (df.days_b_screening_arrest >= -30) &
        (df.is_recid != -1) &
        (df.c_charge_degree != "O") &
        (df.score_text != "N/A")]

# Keep only the columns we need
cols = ["age", "sex", "race", "priors_count", "c_charge_degree",
        "juv_fel_count", "juv_misd_count", "juv_other_count", "two_year_recid"]
df = df[cols].copy()

# Focus on the two groups used in most COMPAS fairness studies
df = df[df.race.isin(["African-American", "Caucasian"])].reset_index(drop=True)

print(df.shape)
print(df.race.value_counts())
print(df.two_year_recid.mean())
df.head()

(5278, 9)
race
African-American    3175
Caucasian           2103
Name: count, dtype: int64
0.47044334975369456


,age,sex,race,priors_count,c_charge_degree,juv_fel_count,juv_misd_count,juv_other_count,two_year_recid
0,34,Male,African-American,0,F,0,0,0,1
1,24,Male,African-American,4,F,0,0,1,1
2,41,Male,Caucasian,14,F,0,0,0,1
3,39,Female,Caucasian,0,M,0,0,0,0
4,27,Male,Caucasian,0,F,0,0,0,0


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

y = df["two_year_recid"]
X = pd.get_dummies(df.drop(columns="two_year_recid"), drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

model_a = LogisticRegression(max_iter=1000)
model_a.fit(X_train, y_train)

pred_a = model_a.predict(X_test)
print("Model A accuracy:", round(accuracy_score(y_test, pred_a), 3))
print(list(X.columns))

Model A accuracy: 0.669
['age', 'priors_count', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'sex_Male', 'race_Caucasian', 'c_charge_degree_M']


In [4]:
X_b = X.drop(columns="race_Caucasian")

X_train_b = X_train.drop(columns="race_Caucasian")
X_test_b = X_test.drop(columns="race_Caucasian")

model_b = LogisticRegression(max_iter=1000)
model_b.fit(X_train_b, y_train)

pred_b = model_b.predict(X_test_b)
print("Model A accuracy:", round(accuracy_score(y_test, pred_a), 3))
print("Model B accuracy:", round(accuracy_score(y_test, pred_b), 3))
print(list(X_b.columns))

Model A accuracy: 0.669
Model B accuracy: 0.666
['age', 'priors_count', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'sex_Male', 'c_charge_degree_M']


In [5]:
from sklearn.metrics import confusion_matrix

race_test = df.loc[X_test.index, "race"]

def group_metrics(y_true, y_pred, race):
    rows = {}
    for g in ["African-American", "Caucasian"]:
        m = (race == g)
        tn, fp, fn, tp = confusion_matrix(y_true[m], y_pred[m]).ravel()
        rows[g] = {
            "selection_rate": (tp + fp) / m.sum(),
            "TPR": tp / (tp + fn),
            "FPR": fp / (fp + tn),
        }
    out = pd.DataFrame(rows).T
    gap = out.loc["African-American"] - out.loc["Caucasian"]
    return out.round(3), gap.round(3)

for name, pred in [("Model A", pred_a), ("Model B", pred_b)]:
    table, gap = group_metrics(y_test, pred, race_test)
    print(f"--- {name} ---")
    print(table)
    print("Gap (African-American minus Caucasian):")
    print(gap.to_string(), "\n")

--- Model A ---
                  selection_rate    TPR    FPR
African-American           0.505  0.677  0.315
Caucasian                  0.232  0.349  0.157
Gap (African-American minus Caucasian):
selection_rate    0.272
TPR               0.328
FPR               0.158 

--- Model B ---
                  selection_rate    TPR    FPR
African-American           0.489  0.659  0.302
Caucasian                  0.250  0.369  0.173
Gap (African-American minus Caucasian):
selection_rate    0.239
TPR               0.290
FPR               0.129 



In [6]:
from sklearn.metrics import roc_auc_score

# Target is now race (1 = African-American, 0 = Caucasian)
race_binary = (df["race"] == "African-American").astype(int)

r_train = race_binary.loc[X_train_b.index]
r_test = race_binary.loc[X_test_b.index]

proxy = LogisticRegression(max_iter=1000)
proxy.fit(X_train_b, r_train)

proba = proxy.predict_proba(X_test_b)[:, 1]
print("Proxy accuracy:", round(accuracy_score(r_test, proxy.predict(X_test_b)), 3))
print("Proxy AUC:", round(roc_auc_score(r_test, proba), 3))
print("Majority-class baseline:", round(1 - r_test.mean(), 3) if r_test.mean() < 0.5 else round(r_test.mean(), 3))

coef = pd.Series(proxy.coef_[0], index=X_train_b.columns).sort_values()
print(coef.round(3))

Proxy accuracy: 0.657
Proxy AUC: 0.676
Majority-class baseline: 0.598
juv_other_count     -0.225
c_charge_degree_M   -0.222
age                 -0.046
juv_misd_count       0.095
priors_count         0.116
sex_Male             0.142
juv_fel_count        0.240
dtype: float64


In [7]:
ta, ga = group_metrics(y_test, pred_a, race_test)
tb, gb = group_metrics(y_test, pred_b, race_test)

summary = pd.DataFrame({
    "Model A (race included)": [accuracy_score(y_test, pred_a), *ga.values],
    "Model B (race removed)":  [accuracy_score(y_test, pred_b), *gb.values],
}, index=["Accuracy", "Selection-rate gap", "TPR gap", "FPR gap"]).round(3)

summary["Change (B - A)"] = (summary.iloc[:, 1] - summary.iloc[:, 0]).round(3)
print(summary)

                    Model A (race included)  Model B (race removed)  \
Accuracy                              0.669                   0.666   
Selection-rate gap                    0.272                   0.239   
TPR gap                               0.328                   0.290   
FPR gap                               0.158                   0.129   

                    Change (B - A)  
Accuracy                    -0.003  
Selection-rate gap          -0.033  
TPR gap                     -0.038  
FPR gap                     -0.029  


In [8]:
# 1. Group sizes
print(df.race.value_counts(), "\n")

# 2. Recidivism rate per race (the label)
print(df.groupby("race")["two_year_recid"].mean().round(3), "\n")

# 3. Average features per race
print(df.groupby("race")[["age", "priors_count", "juv_fel_count",
                          "juv_misd_count", "juv_other_count"]].mean().round(2), "\n")

# 4. Charge degree split per race
print(pd.crosstab(df.race, df.c_charge_degree, normalize="index").round(3))

race
African-American    3175
Caucasian           2103
Name: count, dtype: int64 

race
African-American    0.523
Caucasian           0.391
Name: two_year_recid, dtype: float64 

                    age  priors_count  juv_fel_count  juv_misd_count  \
race                                                                   
African-American  32.43          4.24           0.09            0.14   
Caucasian         37.49          2.29           0.03            0.04   

                  juv_other_count  
race                               
African-American             0.13  
Caucasian                    0.10   

c_charge_degree       F      M
race                          
African-American  0.692  0.308
Caucasian         0.592  0.408


In [9]:
std = X_train_b.std()
influence = (pd.Series(model_b.coef_[0], index=X_train_b.columns) * std).sort_values()
print(influence.round(3))

age                 -0.518
c_charge_degree_M   -0.091
juv_misd_count      -0.035
juv_fel_count       -0.026
sex_Male             0.151
juv_other_count      0.156
priors_count         0.792
dtype: float64


In [10]:
rng = np.random.default_rng(42)
y_arr, p_arr, r_arr = y_test.values, pred_b, race_test.values

def gap_values(y, p, r):
    res = {}
    for g in ["African-American", "Caucasian"]:
        m = (r == g)
        yt, pt = y[m], p[m]
        res[g] = {"sel": pt.mean(), "tpr": pt[yt == 1].mean(), "fpr": pt[yt == 0].mean()}
    return {k: res["African-American"][k] - res["Caucasian"][k] for k in ["sel", "tpr", "fpr"]}

n = len(y_arr)
boot = pd.DataFrame([gap_values(y_arr[i], p_arr[i], r_arr[i])
                     for i in (rng.integers(0, n, n) for _ in range(1000))])

audit = pd.DataFrame({
    "observed_gap": pd.Series(gap_values(y_arr, p_arr, r_arr)),
    "CI_low": boot.quantile(0.025),
    "CI_high": boot.quantile(0.975),
}).round(3)
print(audit)

     observed_gap  CI_low  CI_high
sel         0.239   0.194    0.287
tpr         0.290   0.220    0.364
fpr         0.129   0.071    0.187


In [11]:
# Pick one defendant Model B flagged high risk who did NOT reoffend
flagged = X_test_b[(pred_b == 1) & (y_test.values == 0)]
person = flagged.iloc[[0]]

# How much each feature pushed this person's score
contrib = pd.Series(model_b.coef_[0], index=X_test_b.columns) * (person.iloc[0] - X_train_b.mean())
print("Person's features:")
print(person.T.round(2), "\n")
print("Push on score (positive = toward high risk):")
print(contrib.sort_values().round(3), "\n")
print("Original risk probability:", round(model_b.predict_proba(person)[0, 1], 3))

# What-if: change one thing, priors_count reduced by 2 (floor at 0)
what_if = person.copy()
what_if["priors_count"] = max(0, what_if["priors_count"].iloc[0] - 2)
print("Risk probability if priors_count were 2 lower:", round(model_b.predict_proba(what_if)[0, 1], 3))

Person's features:
                    2631
age                   41
priors_count           7
juv_fel_count          0
juv_misd_count         0
juv_other_count        0
sex_Male            True
c_charge_degree_M  False 

Push on score (positive = toward high risk):
age                  -0.29213
juv_other_count     -0.038827
juv_fel_count        0.003697
juv_misd_count       0.006748
c_charge_degree_M    0.065903
sex_Male             0.074142
priors_count         0.568561
dtype: object 

Original risk probability: 0.569
Risk probability if priors_count were 2 lower: 0.489


In [12]:
from statsmodels.stats.proportion import proportions_ztest

def test_gaps(y, pred, race, label):
    yv = y.values
    aa = (race == "African-American").values
    cc = (race == "Caucasian").values
    groups = {"Selection rate": np.ones(len(yv), bool),
              "TPR": yv == 1,
              "FPR": yv == 0}
    rows = []
    for name, mask in groups.items():
        a, c = mask & aa, mask & cc
        z, p = proportions_ztest([pred[a].sum(), pred[c].sum()],
                                 [a.sum(), c.sum()], alternative="two-sided")
        rows.append({"Model": label, "Metric": name,
                     "Gap": pred[a].mean() - pred[c].mean(),
                     "z": z, "p_value": f"{p:.2e}"})
    return pd.DataFrame(rows)

results = pd.concat([test_gaps(y_test, pred_a, race_test, "A"),
                     test_gaps(y_test, pred_b, race_test, "B")])
print(results.round(3).to_string(index=False))

Model         Metric   Gap      z  p_value
    A Selection rate 0.272 10.874 1.54e-27
    A            TPR 0.328  8.526 1.52e-17
    A            FPR 0.158  5.316 1.06e-07
    B Selection rate 0.239  9.563 1.14e-21
    B            TPR 0.290  7.521 5.42e-14
    B            FPR 0.129  4.346 1.39e-05


In [13]:
from scipy.stats import mannwhitneyu

pos = proba[r_test.values == 1]   # scores for African-American defendants
neg = proba[r_test.values == 0]   # scores for Caucasian defendants

u, p = mannwhitneyu(pos, neg, alternative="two-sided")
print("AUC:", round(u / (len(pos) * len(neg)), 3))
print("p-value:", f"{p:.2e}")

AUC: 0.676
p-value: 1.15e-32


In [14]:
from statsmodels.stats.contingency_tables import mcnemar

a_ok = (pred_a == y_test.values)
b_ok = (pred_b == y_test.values)

table = [[(a_ok & b_ok).sum(),  (a_ok & ~b_ok).sum()],
         [(~a_ok & b_ok).sum(), (~a_ok & ~b_ok).sum()]]

print(pd.DataFrame(table,
                   index=["A correct", "A wrong"],
                   columns=["B correct", "B wrong"]))
print("p-value:", round(mcnemar(table, exact=True).pvalue, 3))

           B correct  B wrong
A correct       1044       15
A wrong           11      514
p-value: 0.557


In [15]:
seeds = [0, 1, 2, 3, 42]
rows = []
for s in seeds:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=s, stratify=y)
    race_te = df.loc[Xte.index, "race"]
    for name, drop in [("A", []), ("B", ["race_Caucasian"])]:
        m = LogisticRegression(max_iter=1000).fit(Xtr.drop(columns=drop), ytr)
        p = m.predict(Xte.drop(columns=drop))
        _, gap = group_metrics(yte, p, race_te)
        rows.append({"seed": s, "model": name,
                     "acc": accuracy_score(yte, p),
                     "sel_gap": gap["selection_rate"],
                     "tpr_gap": gap["TPR"],
                     "fpr_gap": gap["FPR"]})

res = pd.DataFrame(rows)
print(res.round(3).to_string(index=False), "\n")
print(res.groupby("model")[["acc", "sel_gap", "tpr_gap", "fpr_gap"]]
      .agg(["mean", "std"]).round(3))

 seed model   acc  sel_gap  tpr_gap  fpr_gap
    0     A 0.689    0.296    0.328    0.195
    0     B 0.682    0.242    0.274    0.144
    1     A 0.668    0.337    0.402    0.221
    1     B 0.669    0.274    0.327    0.166
    2     A 0.673    0.269    0.284    0.180
    2     B 0.670    0.256    0.266    0.173
    3     A 0.683    0.264    0.289    0.169
    3     B 0.682    0.228    0.255    0.130
   42     A 0.669    0.272    0.328    0.158
   42     B 0.666    0.239    0.290    0.129 

         acc        sel_gap        tpr_gap        fpr_gap       
        mean    std    mean    std    mean    std    mean    std
model                                                           
A      0.676  0.009   0.288  0.030   0.326  0.047   0.185  0.025
B      0.674  0.008   0.248  0.018   0.282  0.028   0.148  0.020


In [16]:
from sklearn.ensemble import RandomForestClassifier

rows = []
for s in seeds:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=s, stratify=y)
    race_te = df.loc[Xte.index, "race"]
    for name, drop in [("A", []), ("B", ["race_Caucasian"])]:
        m = RandomForestClassifier(n_estimators=200, min_samples_leaf=10,
                                   random_state=s, n_jobs=-1)
        m.fit(Xtr.drop(columns=drop), ytr)
        p = m.predict(Xte.drop(columns=drop))
        _, gap = group_metrics(yte, p, race_te)
        rows.append({"seed": s, "model": name,
                     "acc": accuracy_score(yte, p),
                     "sel_gap": gap["selection_rate"],
                     "tpr_gap": gap["TPR"],
                     "fpr_gap": gap["FPR"]})

rf = pd.DataFrame(rows)
print(rf.round(3).to_string(index=False), "\n")
print(rf.groupby("model")[["acc", "sel_gap", "tpr_gap", "fpr_gap"]]
      .agg(["mean", "std"]).round(3))

 seed model   acc  sel_gap  tpr_gap  fpr_gap
    0     A 0.682    0.283    0.308    0.190
    0     B 0.682    0.235    0.257    0.143
    1     A 0.670    0.296    0.350    0.187
    1     B 0.668    0.268    0.316    0.164
    2     A 0.676    0.251    0.266    0.160
    2     B 0.676    0.254    0.278    0.156
    3     A 0.675    0.238    0.243    0.161
    3     B 0.674    0.203    0.193    0.137
   42     A 0.672    0.261    0.294    0.161
   42     B 0.669    0.227    0.270    0.119 

         acc        sel_gap        tpr_gap        fpr_gap       
        mean    std    mean    std    mean    std    mean    std
model                                                           
A      0.675  0.005   0.266  0.024   0.292  0.041   0.172  0.015
B      0.674  0.006   0.237  0.025   0.263  0.045   0.144  0.017


In [17]:
groups = ["African-American", "Caucasian"]
rows = []

for s in seeds:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=s, stratify=y)
    Xtr_b = Xtr.drop(columns="race_Caucasian")
    Xte_b = Xte.drop(columns="race_Caucasian")
    race_tr = df.loc[Xtr.index, "race"].values
    race_te = df.loc[Xte.index, "race"]
    ytr_v = ytr.values

    # 1. Removal (baseline Model B)
    mb = LogisticRegression(max_iter=1000).fit(Xtr_b, ytr)
    pred_rem = mb.predict(Xte_b)

    # 2. Reweighing (race used only to compute weights, not as an input)
    w = np.ones(len(ytr_v))
    for g in groups:
        for lab in [0, 1]:
            mask = (race_tr == g) & (ytr_v == lab)
            w[mask] = ((race_tr == g).mean() * (ytr_v == lab).mean()) / mask.mean()
    mw = LogisticRegression(max_iter=1000).fit(Xtr_b, ytr, sample_weight=w)
    pred_rw = mw.predict(Xte_b)

    # 3. Group thresholds: equalize FPR on the training set (uses race at decision time)
    ptr = mb.predict_proba(Xtr_b)[:, 1]
    target = (ptr[ytr_v == 0] >= 0.5).mean()
    thr = {g: np.quantile(ptr[(race_tr == g) & (ytr_v == 0)], 1 - target) for g in groups}
    pte = mb.predict_proba(Xte_b)[:, 1]
    pred_thr = (pte >= race_te.map(thr).values).astype(int)

    for name, p in [("Removal", pred_rem), ("Reweighing", pred_rw), ("Group thresholds", pred_thr)]:
        _, gap = group_metrics(yte, p, race_te)
        rows.append({"seed": s, "method": name, "acc": accuracy_score(yte, p),
                     "sel_gap": gap["selection_rate"], "tpr_gap": gap["TPR"],
                     "fpr_gap": gap["FPR"]})

rem = pd.DataFrame(rows)
print(rem.round(3).to_string(index=False), "\n")
print(rem.groupby("method")[["acc", "sel_gap", "tpr_gap", "fpr_gap"]].mean().round(3))

 seed           method   acc  sel_gap  tpr_gap  fpr_gap
    0          Removal 0.682    0.242    0.274    0.144
    0       Reweighing 0.684    0.236    0.277    0.129
    0 Group thresholds 0.672    0.102    0.131    0.007
    1          Removal 0.669    0.274    0.327    0.166
    1       Reweighing 0.667    0.279    0.342    0.165
    1 Group thresholds 0.659    0.117    0.181    0.001
    2          Removal 0.670    0.256    0.266    0.173
    2       Reweighing 0.672    0.248    0.262    0.160
    2 Group thresholds 0.658    0.075    0.111   -0.029
    3          Removal 0.682    0.228    0.255    0.130
    3       Reweighing 0.679    0.222    0.240    0.133
    3 Group thresholds 0.669    0.103    0.112    0.023
   42          Removal 0.666    0.239    0.290    0.129
   42       Reweighing 0.674    0.233    0.294    0.111
   42 Group thresholds 0.661    0.072    0.121   -0.040 

                    acc  sel_gap  tpr_gap  fpr_gap
method                                            


In [18]:
import pandas, numpy, sklearn, scipy, statsmodels, sys
print("Python", sys.version.split()[0])
print("pandas", pandas.__version__)
print("numpy", numpy.__version__)
print("scikit-learn", sklearn.__version__)
print("scipy", scipy.__version__)
print("statsmodels", statsmodels.__version__)

Python 3.13.16
pandas 2.2.3
numpy 2.1.3
scikit-learn 1.6.1
scipy 1.16.3
statsmodels 0.15.0
